**Requisitos**

In [9]:
!pip install transformers datasets peft accelerate torch

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch

**Carregar modelo pré-treinado e tokenizador**

In [11]:
model_name = "unicamp-dl/ptt5-base-portuguese-vocab"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Modelo carregado: unicamp-dl/ptt5-base-portuguese-vocab


**Carregar Dataset**

In [12]:
# 1. Carrega o dataset
dataset = load_dataset('json', data_files='dataset_final_divisoes.jsonl')

# 2. Verifica colunas (permanece igual)
print("Colunas originais:", dataset["train"].column_names)

# 3. Cria colunas separadas para o padrão Seq2Seq (mT5)
def convert_to_mt5_format(example):
    # Obtém instrução e resposta tratando variações de nomes de colunas
    instruction = example.get('Instruction', example.get('instruction', ''))
    output = example.get('Output', example.get('output', ''))

    # Para modelos T5/mT5, é comum usar um prefixo para indicar a tarefa.
    # Diferente do Gemma, NÃO usamos chat_template com roles aqui.
    source_text = f"instruction: {instruction}"
    target_text = output

    return {
        "source": source_text,
        "target": target_text
    }

# Aplicamos a conversão criando as duas colunas necessárias
dataset = dataset.map(convert_to_mt5_format)

# 4. Verifica se as colunas foram criadas corretamente
print("Colunas após conversão:", dataset["train"].column_names)
print("\nExemplo de entrada (Source):", dataset["train"]["source"])
print("Exemplo de saída (Target):", dataset["train"]["target"])

# 5. Divide em treino e teste (permanece igual)
dataset = dataset["train"].train_test_split(test_size=0.2)
print("\nDataset final:", dataset)

Colunas originais: ['Instruction', 'Output']
Colunas após conversão: ['Instruction', 'Output', 'source', 'target']

Exemplo de entrada (Source): Column(['instruction: Quais são as partes da Alecrim utilizadas e como elas são preparadas para uso medicinal?', 'instruction: Quais são os cuidados e formas de propagação do Alecrim?', 'instruction: Quais são os usos populares e formas de preparo do Alecrim, além de suas propriedades e efeitos?', 'instruction: Quais são os efeitos e formas de uso do Alecrim, além de suas propriedades anti-inflamatórias e antimicrobianas?', 'instruction: Quais são os usos e formas de preparo do Alecrim?'])
Exemplo de saída (Target): Column(['A Alecrim é uma planta herbácea, perene, aromática, amplamente cultivada. As partes utilizadas incluem folhas, flores e outros tecidos. A preparação para uso medicinal envolve a infusão das folhas em água fervida, geralmente com uma colher de sobremesa, e a aplicação topical de folhas secas ou óleo de folhas em condições d

**Inferência ANTES do Fine-Tuning**

In [13]:
def generate_response_mt5(model, tokenizer, instruction):
    """Gera resposta para o modelo unicamp-dl/ptt5-base-portuguese-vocab (Seq2Seq)."""

    # Modelos T5/mT5 funcionam melhor com um prefixo indicando a tarefa [Histórico]
    prompt = f"Instruction: {instruction}"

    # Tokenização para o Encoder
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad(): # Economiza memória [Histórico]
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=200,
            do_sample=True,
            temperature=0.3
        )

    # Em modelos Seq2Seq, o 'outputs' já contém apenas os tokens gerados pelo Decoder.
    # Não é necessário subtrair o tamanho do prompt [4, 5].
    # Decodifica a primeira (e única) sequência do batch para obter uma string.
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta.strip()

# Exemplo de uso
test_instruction = "Quais são os usos e formas de preparo do Alecrim?"

print("=== TESTE COM unicamp-dl/ptt5-base-portuguese-vocab ===")
print(f"Instrução: {test_instruction}")
# Nota: sem fine-tuning, o mT5-base produzirá saídas aleatórias ou ruído [2]
print(f"Resposta: {generate_response_mt5(base_model, tokenizer, test_instruction)}")

=== TESTE COM unicamp-dl/ptt5-base-portuguese-vocab ===
Instrução: Quais são os usos e formas de preparo do Alecrim?
Resposta: Instruction: Quais são os usos e formas de preparo do Alecrim?


**Configuração LORA**

In [14]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

In [15]:
# Célula auxiliar — verificar os nomes das camadas
for name, module in base_model.named_modules():
    print(name)


shared
encoder
encoder.embed_tokens
encoder.block
encoder.block.0
encoder.block.0.layer
encoder.block.0.layer.0
encoder.block.0.layer.0.SelfAttention
encoder.block.0.layer.0.SelfAttention.q
encoder.block.0.layer.0.SelfAttention.k
encoder.block.0.layer.0.SelfAttention.v
encoder.block.0.layer.0.SelfAttention.o
encoder.block.0.layer.0.SelfAttention.relative_attention_bias
encoder.block.0.layer.0.layer_norm
encoder.block.0.layer.0.dropout
encoder.block.0.layer.1
encoder.block.0.layer.1.DenseReluDense
encoder.block.0.layer.1.DenseReluDense.wi
encoder.block.0.layer.1.DenseReluDense.wo
encoder.block.0.layer.1.DenseReluDense.dropout
encoder.block.0.layer.1.DenseReluDense.act
encoder.block.0.layer.1.layer_norm
encoder.block.0.layer.1.dropout
encoder.block.1
encoder.block.1.layer
encoder.block.1.layer.0
encoder.block.1.layer.0.SelfAttention
encoder.block.1.layer.0.SelfAttention.q
encoder.block.1.layer.0.SelfAttention.k
encoder.block.1.layer.0.SelfAttention.v
encoder.block.1.layer.0.SelfAttentio

In [16]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    !pip install --upgrade torchao>=0.16.0

# Configuração ajustada para mT5 (Seq2Seq)
lora_config = LoraConfig(
    r=8,                          # ← reduzido (menos parâmetros, menos overfitting)
    lora_alpha=16,                # ← alpha = 2*r (padrão)
    target_modules=["q", "v", "k", "o"],  # ← mais camadas
    lora_dropout=0.1,             # ← um pouco mais de dropout
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

# Preparação do modelo
# Certifique-se de que 'base_model' foi carregado com AutoModelForSeq2SeqLM
model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, lora_config)

# Exibe os parâmetros treináveis (deve ser < 1% do total)
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876


**Tokenização**

In [17]:
# Célula 26 - TOKENIZAÇÃO CORRIGIDA PARA mT5

prefix = "Instruction: "

def tokenize_function(examples):
    # Inputs (encoder)
    inputs = [prefix + doc for doc in examples["source"]]
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding=False
    )

    # Labels (decoder) - Para T5/mT5 NÃO precisa do as_target_tokenizer()
    labels = tokenizer(
        examples["target"],
        max_length=512,
        truncation=True,
        padding=False
    )

    # Importante: substituir tokens de padding por -100 nos labels
    # O T5Tokenizer usa o pad_token_id para padding
    labels_ids = labels["input_ids"]

    # Opcional: substituir tokens de padding por -100 (recomendado)
    # Mas o DataCollatorForSeq2Seq já faz isso automaticamente se configurado

    model_inputs["labels"] = labels_ids

    return model_inputs

# Aplica a tokenização ao dataset
print("🔄 Tokenizando dataset...")
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("\n✅ Tokenização concluída!")
print(f"Dataset tokenizado: {tokenized_dataset}")
print(f"\nColunas finais: {tokenized_dataset['train'].column_names}")

🔄 Tokenizando dataset...


Map:   0%|          | 0/132 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]


✅ Tokenização concluída!
Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'source', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 132
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'source', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 34
    })
})

Colunas finais: ['Instruction', 'Output', 'source', 'target', 'input_ids', 'attention_mask', 'labels']


**Data Collator**

In [18]:
from transformers import DataCollatorForSeq2Seq

# Para o mT5, usamos o collator específico para Seq2Seq
# Ele aplicará o padding dinâmico tanto nos inputs quanto nos labels
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,           # ← mude para True (padding dinâmico)
    label_pad_token_id=-100 # ← essencial para ignorar padding no loss
)
print("✅ Data Collator para Seq2Seq configurado")

✅ Data Collator para Seq2Seq configurado


**Argumentos de treinamento**

In [31]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Primeiro, recrie os training arguments com suporte a geração
training_args = Seq2SeqTrainingArguments(
    output_dir="./ptt5-base_plantas_finetuned",
    num_train_epochs=10,                    # ← reduzido de 10
    per_device_train_batch_size=4,         # ← aumente se possível
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,         # ← reduzido
    learning_rate=2e-4,                    # ← reduzido (3e-4 é padrão)
    warmup_ratio=0.2,                      # ← volta para ratio (mais estável)
    weight_decay=0.01,                     # ← reduzido
    max_grad_norm=1.0,
    label_smoothing_factor=0.05,           # ← reduzido (0.1 foi agressivo)
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    report_to="none",
    save_total_limit=2,
    logging_steps=10,
    # ← Remova predict_with_generate daqui (não funciona no TrainingArguments)
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


**Iniciar o trainer**

In [35]:
import torch

# Agora use o Seq2SeqTrainer (não o Trainer padrão)
# É essencial desabilitar `use_cache` quando `gradient_checkpointing=True` para evitar `CheckpointError`
# Para PeftModel, é mais robusto desabilitar no base_model
if hasattr(model, 'base_model') and hasattr(model.base_model, 'config'):
    model.base_model.config.use_cache = False
else:
    model.config.use_cache = False # Fallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

print("✅ Trainer configurado")
print(f"   Amostras de treino:    {len(tokenized_dataset['train'])}")
print(f"   Amostras de validação: {len(tokenized_dataset['test'])}")
print(f"   Épocas:                {training_args.num_train_epochs}")

✅ Trainer configurado
   Amostras de treino:    132
   Amostras de validação: 34
   Épocas:                20


**Treinar modelo**

In [36]:
# Célula — Fix para erro de VideoReader
import sys

# Remove o módulo problemático para evitar o erro de importação
if 'torchvision' in sys.modules:
    del sys.modules['torchvision']

# Reinstala versões compatíveis
!pip install -q --upgrade datasets
!pip install -q torchvision --extra-index-url https://download.pytorch.org/whl/cu118

# Reimporta o que precisa
from datasets import load_dataset
import torch

print("✅ Dependências corrigidas")
print(f"✅ Torch: {torch.__version__}")

✅ Dependências corrigidas
✅ Torch: 2.11.0+cu128


In [37]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,6.874889,3.211326
2,6.843328,3.190929
3,6.814925,3.172740
4,6.294252,3.158745
5,6.611341,3.136177
6,6.613864,3.124097
7,6.277005,3.110783
8,6.582915,3.093750
9,6.419201,3.084482
10,6.041105,3.075442


TrainOutput(global_step=340, training_loss=6.339762003281537, metrics={'train_runtime': 282.1394, 'train_samples_per_second': 9.357, 'train_steps_per_second': 1.205, 'total_flos': 101442161221632.0, 'train_loss': 6.339762003281537, 'epoch': 20.0})

**Gráfico de treinamento**

In [ ]:

def extrair_logs_por_epoca(trainer):
    """Extrai losses por época (mais fácil de visualizar)."""

    logs = trainer.state.log_history

    epochs = []
    train_losses = []
    eval_losses = []

    current_epoch = 1
    current_train_loss = None

    for entry in logs:
        # Training loss (tem 'epoch' e 'loss')
        if 'loss' in entry and 'epoch' in entry and 'eval_loss' not in entry:
            current_train_loss = entry['loss']

        # Validation loss (marca o fim de uma época)
        if 'eval_loss' in entry and 'epoch' in entry:
            epochs.append(int(entry['epoch']))
            eval_losses.append(entry['eval_loss'])
            if current_train_loss is not None:
                train_losses.append(current_train_loss)
                current_train_loss = None

    return epochs, train_losses, eval_losses

In [ ]:
def plotar_loss_por_epoca(trainer, nome_modelo):
    """Plota loss por época (mais limpo que por step)."""

    epochs, train_losses, eval_losses = extrair_logs_por_epoca(trainer)

    plt.figure(figsize=(10, 6))

    # Training loss
    if train_losses and len(train_losses) == len(epochs):
        plt.plot(epochs, train_losses,
                 label="Training Loss",
                 color="blue",
                 linewidth=2,
                 marker='s',
                 markersize=6)

    # Validation loss
    if eval_losses:
        plt.plot(epochs, eval_losses,
                 label="Validation Loss",
                 color="orange",
                 linewidth=2.5,
                 marker='o',
                 markersize=7)

    plt.title(f"Evolução do Loss por Época — {nome_modelo}", fontsize=14, fontweight="bold")
    plt.xlabel("Época", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.xticks(epochs)  # Mostra todas as épocas
    plt.tight_layout()
     # Salva o gráfico
    nome_arquivo = f"loss_epochs_{nome_modelo.replace('/', '_').replace(' ', '_')}.png"
    plt.savefig(nome_arquivo, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Gráfico salvo como loss_{nome_modelo.replace(' ', '_')}.png")

# Uso após treinar cada modelo
plotar_loss_por_epoca(trainer, "google/mt5-small")

**Salvar modelo**

In [ ]:
model.save_pretrained("lora_seq2seq_model_2/")
tokenizer.save_pretrained("mt5_small_tokenizer")

**Inferência APÓS o Fine-Tuning**

In [ ]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
from transformers import AutoModelForSeq2SeqLM

finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("lora_seq2seq_model_2/")
finetuned_tokenizer = AutoTokenizer.from_pretrained("mt5_small_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

In [ ]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response_mt5(finetuned_model, finetuned_tokenizer, test_instruction)}")